In [2]:
import os
import numpy as np
import keras
from keras import layers, regularizers, callbacks
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import train_test_split
import pandas as pd
from joblib import Parallel, delayed
from tensorflow.keras import layers, regularizers, callbacks

In [ ]:
# Load dataset
df = pd.read_csv("data.csv")

print("Columns:", df.columns)
print("Missing values:\n", df.isnull().sum())

X = df[['HD', 'DBT', 'RH', 'DSR', 'DiSR', 'GHR', 'LA']].values
y = df[['Energy']].values


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

In [ ]:
# Hyperparameters
hidden_layers = [
    [512, 256, 128], [256, 128], 
    [512, 256, 64], [256, 128, 64],
    [1024, 512, 256]
]
dropouts = [(0.05, 0.2), (0.1, 0.3)]
batch_sizes = [64, 128]
lr_range = [0.001, 0.005]
l2_range = [0.001, 0.01]

# Results storage
results = defaultdict(list)
best_r2 = -np.inf
best_model = None
best_config = None

# Train models and evaluate its performance
def train_and_evaluate_model(hidden, dropout1, dropout2, batch_size, lr, l2_reg):
    global best_r2_ev, best_model, best_config
    config_str = f"_HL{hidden}_dr1{dropout1}_dr2{dropout2}_bs{batch_size}_lr{lr}_l2{l2_reg}"
    print(f"\n🔁 Training Model: {config_str}")

    # Model
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    for i, units in enumerate(hidden):
        model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout1 if i == 0 else dropout2))
    model.add(layers.Dense(1))

    
    # Compile
    optimizer = keras.optimizers.Adam(
        learning_rate=keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=lr,
            decay_steps=100,
            decay_rate=0.9
        )
    )
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    # Train
    early_stop = callbacks.EarlyStopping(patience=25, restore_best_weights=True, monitor='val_loss')
    history = model.fit(
        X_train_scaled, y_train_scaled,
        epochs=300,
        batch_size=batch_size,
        validation_data=(X_test_scaled, y_test_scaled),
        callbacks=[early_stop],
        verbose=0
    )

    # Predict
    y_pred_scaled = model.predict(X_test_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled)

    # Evaluate
    r2_model = r2_score(y_test, y_pred)
    mae_model = mean_absolute_error(y_test, y_pred)
    mse_model = mean_squared_error(y_test, y_pred)

    # Save model
    model_filename = f"saved_models/{config_str}.h5"
    model.save(model_filename)
    
    return {
    'config': config_str,
    'r2_model': r2_model,
    'mae_model': mae_model,
    'mse_model': mse_model
    }



tasks = []
for hidden in hidden_layers:
    for dropout1, dropout2 in dropouts:
        for batch_size in batch_sizes:
            for lr in lr_range:
                for l2_reg in l2_range:
                    tasks.append(delayed(train_and_evaluate_model)(
                        hidden, dropout1, dropout2,
                        batch_size, lr, l2_reg
                        ))

# Run the tasks and collect results
results_list = Parallel(n_jobs=-1, verbose=10)(tasks)

results = defaultdict(list)
for res in results_list:
    for key, value in res.items():
        results[key].append(value)

print(f"\nResults saved to architecture_comparison.xlsx")
print(f"All models saved in /saved_models/")
print(f"Best model (R² for Ev = {best_r2_ev:.4f}): {best_config}")

results_df = pd.DataFrame(results)
results_df.to_excel("architecture_comparison.xlsx", index=False)
results_df = pd.DataFrame(results)
results_df.to_csv("architecture_comparison.csv", index=False)
